In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load the CSV from the notebook's current folder
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Inspect the data
print("First five records:")
print(df.head())
print("\nDataset shape:", df.shape)
df.info()
print("\nMissing values:")
print(df.isna().sum())
print("\nStatistical summary:")
print(df.describe(include="all"))

# Trim whitespace in text columns
text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].str.strip()

# Convert TotalCharges; blank or invalid values become missing
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Fill missing values
for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode = df[col].mode()
            if not mode.empty:
                df[col] = df[col].fillna(mode.iloc[0])

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Report potential numeric outliers (does not remove them)
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    outlier_count = (
        (df[col] < q1 - 1.5 * iqr) |
        (df[col] > q3 + 1.5 * iqr)
    ).sum()
    print(f"{col}: {outlier_count} potential outliers")

# Add a tenure group if the column exists
if "tenure" in df.columns:
    df["TenureGroup"] = pd.cut(
        df["tenure"],
        bins=[-1, 12, 24, 48, np.inf],
        labels=["0-12", "13-24", "25-48", "49+"]
    )

# Add standardized copies of selected numeric columns
scale_cols = [
    col for col in ["tenure", "MonthlyCharges", "TotalCharges"]
    if col in df.columns
]

if scale_cols:
    scaler = StandardScaler()
    scaled_cols = [f"{col}_Scaled" for col in scale_cols]
    df[scaled_cols] = scaler.fit_transform(df[scale_cols])

# Review and save
print("\nCleaned dataset:")
print(df.head())
print("\nRemaining missing values:")
print(df.isna().sum())

output_file = "Cleaned_Telecom_Customer_Churn.csv"
df.to_csv(output_file, index=False)
print(f"\nCleaned dataset saved as {output_file}")

First five records:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies 